# Day 3: DataFrame Practice Exercises

## Welcome!
These exercises are designed for beginners to practice PySpark DataFrame operations using the COVID-19 dataset. Follow each step carefully to build your skills.

## Before You Start
- Run `docker-compose up` in the `01_basic_spark` directory.
- Open Jupyter at `http://localhost:8888`.
- Place `covid-data.csv` in the `covid-dataset/` directory.
- Download the dataset if needed: https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv

## Exercises



In [1]:
from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder.appName("SparkDataFrameExercises").getOrCreate()

In [4]:
import pyspark.sql.functions as sf
f = F = sf

In [2]:
df_covid = _df_covid = spark.read.csv("data/owid-covid-data.csv", header=True, inferSchema=True)

---

# Exercise 15: Sort by Multiple Columns
#### What to Do
- Sort the DataFrame by `continent` (ascending) and `total_cases` (descending).

#### Steps
- Load the CSV into a DataFrame.
- Use `orderBy()` with `continent` and `total_cases`.
- Show 10 rows with `continent`, `location`, `total_cases`.


## Solution without DISTINCT 

As I would expect. Order appears maintained in output

**Note:** Select before orderBy to be consistent to steps below

In [39]:
result_df1 = (
    df_covid
    .filter(sf.col("continent").isNotNull())
    .select(
        "continent",
        "location",
        "total_cases",
    )
    .orderBy(
        sf.col("continent").asc(),
        sf.col("total_cases").desc(),
    )
)
result_df1.show(10)

+---------+------------+-----------+
|continent|    location|total_cases|
+---------+------------+-----------+
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072763|
|   Africa|South Africa|    4072763|
+---------+------------+-----------+
only showing top 10 rows



In [40]:
#result_df1.explain()
result_df1.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (5)
+- Sort (4)
   +- Exchange (3)
      +- Filter (2)
         +- Scan csv  (1)


(1) Scan csv 
Output [3]: [continent#18, location#19, total_cases#21]
Batched: false
Location: InMemoryFileIndex [file:/home/jovyan/work/data/owid-covid-data.csv]
PushedFilters: [IsNotNull(continent)]
ReadSchema: struct<continent:string,location:string,total_cases:int>

(2) Filter
Input [3]: [continent#18, location#19, total_cases#21]
Condition : isnotnull(continent#18)

(3) Exchange
Input [3]: [continent#18, location#19, total_cases#21]
Arguments: rangepartitioning(continent#18 ASC NULLS FIRST, total_cases#21 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=1256]

(4) Sort
Input [3]: [continent#18, location#19, total_cases#21]
Arguments: [continent#18 ASC NULLS FIRST, total_cases#21 DESC NULLS LAST], true, 0

(5) AdaptiveSparkPlan
Output [3]: [continent#18, location#19, total_cases#21]
Arguments: isFinalPlan=false




## Solution with DISTINCT after OrderBy

Order appears scrambled in output

**Note:** select moved to make ORDER BY and DISTINCT work on the same 3 target columns

Same result and operations as given in the reference solution [03_solution.ipynb](https://github.com/neuefische/de-week-9-Batch-Processing-Pyspark/blob/main/03_spark_dataframe/solutions/03_solution.ipynb) (see exact reference solution below for comparison)


In [41]:
result_df2 = (
    df_covid
    .filter(sf.col("continent").isNotNull())
    .select(
        "continent",
        "location",
        "total_cases",
    )
    .orderBy(
        sf.col("continent").asc(),
        sf.col("total_cases").desc(),
    )
    .distinct()
)
result_df2.show(10)

+---------+-----------+-----------+
|continent|   location|total_cases|
+---------+-----------+-----------+
|     Asia|Afghanistan|      57793|
|   Europe|    Albania|     334596|
|   Africa|    Algeria|     272046|
|   Europe|    Andorra|        466|
|   Europe|    Andorra|      41013|
|   Africa|     Angola|      65011|
|  Oceania|  Australia|       6289|
|  Oceania|  Australia|    4167400|
|  Oceania|  Australia|    8292337|
|   Europe|    Austria|      19955|
+---------+-----------+-----------+
only showing top 10 rows



## Question

Looking at the `result_df2.explain()` below the `Sort` operation used in the pipeline
without distinct (see above `result_df1.explain()` ) is not included.

Is the data not sorted, because Spark knows 
it will shuffle it again for the distinct() operation?


In [49]:
#result_df2.explain()
result_df2.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Filter (2)
            +- Scan csv  (1)


(1) Scan csv 
Output [3]: [continent#18, location#19, total_cases#21]
Batched: false
Location: InMemoryFileIndex [file:/home/jovyan/work/data/owid-covid-data.csv]
PushedFilters: [IsNotNull(continent)]
ReadSchema: struct<continent:string,location:string,total_cases:int>

(2) Filter
Input [3]: [continent#18, location#19, total_cases#21]
Condition : isnotnull(continent#18)

(3) HashAggregate
Input [3]: [continent#18, location#19, total_cases#21]
Keys [3]: [continent#18, location#19, total_cases#21]
Functions: []
Aggregate Attributes: []
Results [3]: [continent#18, location#19, total_cases#21]

(4) Exchange
Input [3]: [continent#18, location#19, total_cases#21]
Arguments: hashpartitioning(continent#18, location#19, total_cases#21, 200), ENSURE_REQUIREMENTS, [plan_id=1332]

(5) HashAggregate
Input [3]: [continent#18, location#19, 

### Reference solution for comparison

In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Initialize Spark session
spark = SparkSession.builder.appName("DFEx15").getOrCreate()

# Load CSV with header and infer schema
df = (spark.read
      .option("header","true")
      .option("inferSchema","true")
      #.csv("covid-dataset/covid-data.csv")
      .csv("data/owid-covid-data.csv")
     )

# Keep rows with valid continent
df = df.filter(col("continent").isNotNull())

# Order by continent ascending & total_cases descending, select needed columns, remove duplicates
df = (
    df
    .orderBy(
        col("continent").asc(),
        col("total_cases").desc(),
    )
    .select("continent","location","total_cases")
    .distinct()
)

# Show top 10 results
df.show(10)


+---------+-----------+-----------+
|continent|   location|total_cases|
+---------+-----------+-----------+
|     Asia|Afghanistan|      57793|
|   Europe|    Albania|     334596|
|   Africa|    Algeria|     272046|
|   Europe|    Andorra|        466|
|   Europe|    Andorra|      41013|
|   Africa|     Angola|      65011|
|  Oceania|  Australia|       6289|
|  Oceania|  Australia|    4167400|
|  Oceania|  Australia|    8292337|
|   Europe|    Austria|      19955|
+---------+-----------+-----------+
only showing top 10 rows



## Solution with DISTINCT **without** OrderBy

Solution and plan seem to be identical to `result_df2` above **with** the orderBy before distinct


In [50]:
result_df2a = (
    df_covid
    .filter(sf.col("continent").isNotNull())
    .select(
        "continent",
        "location",
        "total_cases",
    )
    .distinct()
)
result_df2a.show(10)

+---------+-----------+-----------+
|continent|   location|total_cases|
+---------+-----------+-----------+
|     Asia|Afghanistan|      57793|
|   Europe|    Albania|     334596|
|   Africa|    Algeria|     272046|
|   Europe|    Andorra|        466|
|   Europe|    Andorra|      41013|
|   Africa|     Angola|      65011|
|  Oceania|  Australia|       6289|
|  Oceania|  Australia|    4167400|
|  Oceania|  Australia|    8292337|
|   Europe|    Austria|      19955|
+---------+-----------+-----------+
only showing top 10 rows



In [51]:
#result_df2a.explain()
result_df2a.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Filter (2)
            +- Scan csv  (1)


(1) Scan csv 
Output [3]: [continent#18, location#19, total_cases#21]
Batched: false
Location: InMemoryFileIndex [file:/home/jovyan/work/data/owid-covid-data.csv]
PushedFilters: [IsNotNull(continent)]
ReadSchema: struct<continent:string,location:string,total_cases:int>

(2) Filter
Input [3]: [continent#18, location#19, total_cases#21]
Condition : isnotnull(continent#18)

(3) HashAggregate
Input [3]: [continent#18, location#19, total_cases#21]
Keys [3]: [continent#18, location#19, total_cases#21]
Functions: []
Aggregate Attributes: []
Results [3]: [continent#18, location#19, total_cases#21]

(4) Exchange
Input [3]: [continent#18, location#19, total_cases#21]
Arguments: hashpartitioning(continent#18, location#19, total_cases#21, 200), ENSURE_REQUIREMENTS, [plan_id=1606]

(5) HashAggregate
Input [3]: [continent#18, location#19, 

## Solution with DISTINCT before orderBy 

select moved to make DISTINCT work only on the 3 target columns

In [45]:
result_df3 = (
    df_covid
    .filter(sf.col("continent").isNotNull())
    .select(
        "continent",
        "location",
        "total_cases",
    )
    .distinct()
    .orderBy(
        sf.col("continent").asc(),
        sf.col("total_cases").desc(),
    )
)
result_df3.show(10)

+---------+------------+-----------+
|continent|    location|total_cases|
+---------+------------+-----------+
|   Africa|South Africa|    4072765|
|   Africa|South Africa|    4072763|
|   Africa|South Africa|    4072759|
|   Africa|South Africa|    4072751|
|   Africa|South Africa|    4072744|
|   Africa|South Africa|    4072740|
|   Africa|South Africa|    4072726|
|   Africa|South Africa|    4072720|
|   Africa|South Africa|    4072712|
|   Africa|South Africa|    4072708|
+---------+------------+-----------+
only showing top 10 rows



## Observations

Using distinct before orderBy appears to trigger more operations in the plan.

Looking at `result_df2.explain()` I see the 3 steps 

```
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
```

that are appear in this plan as well.

Additionally there are an additional `Exchange + Sort` indicating the orderBy is actually happening.

In [48]:
#result_df3.explain()
result_df3.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Exchange (4)
            +- HashAggregate (3)
               +- Filter (2)
                  +- Scan csv  (1)


(1) Scan csv 
Output [3]: [continent#18, location#19, total_cases#21]
Batched: false
Location: InMemoryFileIndex [file:/home/jovyan/work/data/owid-covid-data.csv]
PushedFilters: [IsNotNull(continent)]
ReadSchema: struct<continent:string,location:string,total_cases:int>

(2) Filter
Input [3]: [continent#18, location#19, total_cases#21]
Condition : isnotnull(continent#18)

(3) HashAggregate
Input [3]: [continent#18, location#19, total_cases#21]
Keys [3]: [continent#18, location#19, total_cases#21]
Functions: []
Aggregate Attributes: []
Results [3]: [continent#18, location#19, total_cases#21]

(4) Exchange
Input [3]: [continent#18, location#19, total_cases#21]
Arguments: hashpartitioning(continent#18, location#19, total_cases#21, 200), ENSURE_REQUIREMENTS, [plan_id=152